# Clean V2 RidgeLinUCB Evaluation

This notebook is the active RidgeLinUCB benchmark for the cleaned V2 preprocessing flow. It evaluates only the feature sets that survived the previous broad feature-engineering round:

1. `v2_pharmacogenetic_augmented_regression_hw`
2. `v2_strict_regression_hw_full_dummies`
3. `v2_strict_knn_hw_full_dummies`

Retired exploratory feature sets are intentionally not run here: group-median height/weight, drop-first dummies, expanded target-INR/indication, and IWPC-minimal features.


In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    for candidate in [path] + list(path.parents):
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not find repository root.')


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.bandits.linUCB import RidgeLinUCB
from src.evaluation.feature_set_benchmark import (
    FINAL_FEATURE_SETS,
    FINAL_RIDGE_GRID,
    available_final_feature_sets,
    classwise_metrics,
    confusion_table,
    evaluate_feature_set_grid,
    list_feature_sets,
    load_v2_assets,
    plot_feature_set_accuracy,
    save_benchmark_outputs,
    static_policy_summary,
    summarize_error_metrics,
)

RESULTS_DIR = REPO_ROOT / 'results' / 'v2_feature_set_evaluation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Repo root:', REPO_ROOT)
print('Results dir:', RESULTS_DIR)


## 1. Load cleaned V2 assets

Run `src/preprocessing/preprocess_v2.ipynb` first. The feature manifest should now contain only the three active feature sets.


In [ ]:
df_v2, feature_sets, preprocess_output_dir = load_v2_assets(REPO_ROOT)
feature_set_names = available_final_feature_sets(feature_sets, FINAL_FEATURE_SETS)

print('V2 modeling table:', df_v2.shape)
print('V2 preprocessing output:', preprocess_output_dir)
print('Active feature sets:', feature_set_names)

list_feature_sets(feature_sets)


## 2. Static baseline checks

These are not bandits, but they are the key reference lines for the project: fixed medium dose, clinical formula, and pharmacogenetic formula.


In [ ]:
static_baselines = static_policy_summary(df_v2, feature_set_names)
static_baselines.to_csv(RESULTS_DIR / 'static_baseline_summary.csv', index=False)
static_baselines.sort_values(['feature_set', 'accuracy'], ascending=[True, False])


## 3. RidgeLinUCB final grid

This compact grid keeps the values that were consistently useful in the previous sweep. It includes `alpha=0` to check whether explicit UCB exploration is helping beyond regularized greedy learning.


In [ ]:
SEEDS = range(5)
RIDGE_ALPHAS = FINAL_RIDGE_GRID['alphas']
RIDGE_LAMBDAS = FINAL_RIDGE_GRID['lambda_regs']

ridge_summary, ridge_results, ridge_scale_stats = evaluate_feature_set_grid(
    df=df_v2,
    feature_sets=feature_sets,
    linucb_cls=RidgeLinUCB,
    feature_set_names=feature_set_names,
    alphas=RIDGE_ALPHAS,
    lambda_regs=RIDGE_LAMBDAS,
    seeds=SEEDS,
    standardize=True,
    progress=False,
    verbose=True,
)

ridge_summary.head(15)


## 4. Save metrics

The saved files include leaderboard, step-level trajectories, class-wise metrics, severe/one-step error summaries, and confusion matrices.


In [ ]:
ridge_error_summary = summarize_error_metrics(ridge_results)
ridge_classwise = classwise_metrics(ridge_results)
ridge_confusion_true = confusion_table(ridge_results, normalize='true')
ridge_confusion_counts = confusion_table(ridge_results, normalize=None)

ridge_summary_with_stage = ridge_summary.copy()
ridge_summary_with_stage['stage'] = 'clean_final_grid'

written = save_benchmark_outputs(
    RESULTS_DIR,
    combined_feature_set_leaderboard=ridge_summary_with_stage,
    ridge_feature_set_step_results=ridge_results,
    ridge_feature_set_scale_stats=ridge_scale_stats,
    ridge_feature_set_error_summary=ridge_error_summary,
    ridge_feature_set_classwise=ridge_classwise,
    ridge_feature_set_confusion_true_normalized=ridge_confusion_true,
    ridge_feature_set_confusion_counts=ridge_confusion_counts,
    static_baseline_summary=static_baselines,
)
written


## 5. Leaderboard and diagnostics

The first table is the main RidgeLinUCB leaderboard. The second table shows the error profile for the same configurations.


In [ ]:
ridge_summary.head(20)


In [ ]:
ridge_error_summary.head(20)


In [ ]:
best = ridge_summary.iloc[0]
best_filter = (
    (ridge_classwise['feature_set'] == best['feature_set'])
    & (ridge_classwise['alpha'] == best['alpha'])
    & (ridge_classwise['lambda_reg'] == best['lambda_reg'])
)
ridge_classwise.loc[best_filter].sort_values('class_id')


In [ ]:
fig, ax = plot_feature_set_accuracy(ridge_summary, top_n=15, title='Clean V2 RidgeLinUCB leaderboard')
fig.savefig(RESULTS_DIR / 'clean_v2_ridge_leaderboard.png', dpi=180, bbox_inches='tight')
plt.show()


## 6. Interpretation checkpoint

Use this notebook to choose the active feature set for advanced algorithms. Based on previous runs, the expected winner is the pharmacogenetic-augmented regression-height/weight feature set, with the strict regression-height/weight set as the clean non-baseline-augmented comparison.
